<a href="https://colab.research.google.com/github/Indradumnabanerji/DataScienceTutorials/blob/main/HierarchialData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
from collections import defaultdict

# Re-using the flatten_hierarchy function from the previous immersive
def flatten_hierarchy(hierarchical_data):
    """
    Converts a list of hierarchical data records into a flattened, generic structure.

    Each input record (dictionary) is expected to have a 'Node_PK' and
    multiple 'Level X[A-Z]' columns (e.g., 'Level 1A', 'Level 2B').

    The output will be a list of dictionaries, where the level columns are
    renamed to a generic 'LevelX' format (e.g., 'Level1', 'Level2').

    Args:
        hierarchical_data (list of dict): A list of dictionaries, where each
                                         dictionary represents a row of hierarchical data.

    Returns:
        list of dict: A list of flattened dictionaries.
    """
    flattened_records = []

    # Regex to extract the level number from column names like 'Level 1A', 'Level 2B'
    level_pattern = re.compile(r'Level (\d+)[A-Z]?')

    for record in hierarchical_data:
        flattened_record = {}
        for key, value in record.items():
            if 'Node_PK' in key: # Handle Node_PK, Node_PK1, Node_PK2 etc.
                flattened_record[key] = value
            else:
                match = level_pattern.match(key)
                if match:
                    # Extract the level number and create a generic key 'LevelX'
                    level_number = match.group(1)
                    generic_level_key = f'Level{level_number}'
                    flattened_record[generic_level_key] = value
                else:
                    # If a key doesn't match the pattern (e.g., an unexpected column),
                    # include it as is.
                    flattened_record[key] = value
        flattened_records.append(flattened_record)

    return flattened_records

def create_lookup_dict(flattened_data, pk_column_name):
    """
    Converts flattened hierarchy data into a lookup dictionary.

    Args:
        flattened_data (list of dict): Output from flatten_hierarchy.
        pk_column_name (str): The name of the primary key column (e.g., 'Node_PK1').

    Returns:
        dict: A dictionary where key is the PK value and value is a dict of its levels.
    """
    lookup = {}
    for record in flattened_data:
        pk_value = record[pk_column_name]
        # Create a new dict containing only the level data, excluding the PK itself
        level_data = {k: v for k, v in record.items() if k != pk_column_name}
        lookup[pk_value] = level_data
    return lookup

def merge_and_pivot_hierarchies(hierarchy_lookups, metrics_data):
    """
    Merges hierarchical level data with metrics data and pivots metrics.

    Args:
        hierarchy_lookups (dict): A dictionary mapping PK column names (e.g., 'Node_PK1')
                                  to their respective lookup dictionaries.
                                  Example: {'Node_PK1': {'A1': {'Level1': 'A', ...}}, ...}
        metrics_data (list of dict): The raw metrics hierarchy data.

    Returns:
        list of dict: The final flattened and merged hierarchy.
    """
    # Group metrics by their foreign keys to handle pivoting
    grouped_metrics = defaultdict(lambda: {'metrics': {}})
    for row in metrics_data:
        fk1 = row['Node_FK1']
        fk2 = row['Node_FK2']
        fk3 = row['Node_FK3']
        metric_name = row['Metrics']
        metric_value = row['Value']

        # Use a tuple of foreign keys as the composite key for grouping
        composite_key = (fk1, fk2, fk3)
        grouped_metrics[composite_key]['metrics'][metric_name] = metric_value

        # Store FKs for later use (they are common for the group)
        grouped_metrics[composite_key]['Node_FK1'] = fk1
        grouped_metrics[composite_key]['Node_FK2'] = fk2
        grouped_metrics[composite_key]['Node_FK3'] = fk3

    final_hierarchy = []

    for composite_key, data in grouped_metrics.items():
        fk1 = data['Node_FK1']
        fk2 = data['Node_FK2']
        fk3 = data['Node_FK3']
        metrics_values = data['metrics']

        # Initialize the new record with the foreign keys (can be removed later if not needed in final output)
        new_record = {}

        # Look up and add Level data from Hierarchy 1
        if fk1 in hierarchy_lookups.get('Node_PK1', {}):
            for k, v in hierarchy_lookups['Node_PK1'][fk1].items():
                new_record[f'{k}A'] = v # Append 'A' to level names for Hierarchy 1

        # Look up and add Level data from Hierarchy 2
        if fk2 in hierarchy_lookups.get('Node_PK2', {}):
            for k, v in hierarchy_lookups['Node_PK2'][fk2].items():
                new_record[f'{k}B'] = v # Append 'B' to level names for Hierarchy 2

        # Look up and add Level data from Hierarchy 3
        if fk3 in hierarchy_lookups.get('Node_PK3', {}):
            for k, v in hierarchy_lookups['Node_PK3'][fk3].items():
                new_record[f'{k}C'] = v # Append 'C' to level names for Hierarchy 3

        # Add the pivoted metrics
        for metric_key, metric_val in metrics_values.items():
            new_record[metric_key] = metric_val

        final_hierarchy.append(new_record)

    return final_hierarchy

# --- Your Provided Hierarchical Data ---

# Hierarchy 1 (Node_PK1)
hierarchy1_data = [
    {'Node_PK1': 'A1', 'Level 1A': 'A', 'Level 2A': 'E', 'Level 3A': 'I'},
    {'Node_PK1': 'A2', 'Level 1A': 'B', 'Level 2A': 'F', 'Level 3A': 'J'},
    {'Node_PK1': 'A3', 'Level 1A': 'C', 'Level 2A': 'G', 'Level 3A': 'K'},
    {'Node_PK1': 'A4', 'Level 1A': 'D', 'Level 2A': 'H', 'Level 3A': 'L'},
]

# Hierarchy 2 (Node_PK2)
hierarchy2_data = [
    {'Node_PK2': 'B1', 'Level 1B': 'A', 'Level 2B': 'E', 'Level 3B': 'I'},
    {'Node_PK2': 'B2', 'Level 1B': 'B', 'Level 2B': 'F', 'Level 3B': 'J'},
    {'Node_PK2': 'B3', 'Level 1B': 'C', 'Level 2B': 'G', 'Level 3B': 'K'},
    {'Node_PK2': 'B4', 'Level 1B': 'D', 'Level 2B': 'H', 'Level 3B': 'L'},
]

# Hierarchy 3 (Node_PK3)
hierarchy3_data = [
    {'Node_PK3': 'C1', 'Level 1C': 'A', 'Level 2C': 'E', 'Level 3C': 'I'},
    {'Node_PK3': 'C2', 'Level 1C': 'B', 'Level 2C': 'F', 'Level 3C': 'J'},
    {'Node_PK3': 'C3', 'Level 1C': 'C', 'Level 2C': 'G', 'Level 3C': 'K'},
    {'Node_PK3': 'C4', 'Level 1C': 'D', 'Level 2C': 'H', 'Level 3C': 'L'},
]

# Metrics Hierarchy
metrics_data = [
    {'Node_FK1': 'A1', 'Node_FK2': 'B1', 'Node_FK3': 'C1', 'Metrics': 'M1', 'Value': 30},
    {'Node_FK1': 'A1', 'Node_FK2': 'B1', 'Node_FK3': 'C1', 'Metrics': 'M2', 'Value': 20},
    {'Node_FK1': 'A1', 'Node_FK2': 'B1', 'Node_FK3': 'C1', 'Metrics': 'M3', 'Value': 30},
    {'Node_FK1': 'A1', 'Node_FK2': 'B1', 'Node_FK3': 'C1', 'Metrics': 'M4', 'Value': 49},
    {'Node_FK1': 'A2', 'Node_FK2': 'B2', 'Node_FK3': 'C2', 'Metrics': 'M1', 'Value': 50},
    {'Node_FK1': 'A2', 'Node_FK2': 'B2', 'Node_FK3': 'C2', 'Metrics': 'M2', 'Value': 80},
    {'Node_FK1': 'A2', 'Node_FK2': 'B2', 'Node_FK3': 'C2', 'Metrics': 'M3', 'Value': 70},
    {'Node_FK1': 'A2', 'Node_FK2': 'B2', 'Node_FK3': 'C2', 'Metrics': 'M4', 'Value': 49},
    {'Node_FK1': 'A3', 'Node_FK2': 'B3', 'Node_FK3': 'C3', 'Metrics': 'M1', 'Value': 70},
    {'Node_FK1': 'A3', 'Node_FK2': 'B3', 'Node_FK3': 'C3', 'Metrics': 'M2', 'Value': 90},
    {'Node_FK1': 'A3', 'Node_FK2': 'B3', 'Node_FK3': 'C3', 'Metrics': 'M3', 'Value': 70},
    {'Node_FK1': 'A3', 'Node_FK2': 'B3', 'Node_FK3': 'C3', 'Metrics': 'M4', 'Value': 59},
    {'Node_FK1': 'A4', 'Node_FK2': 'B4', 'Node_FK3': 'C4', 'Metrics': 'M1', 'Value': 30},
    {'Node_FK1': 'A4', 'Node_FK2': 'B4', 'Node_FK3': 'C4', 'Metrics': 'M2', 'Value': 80},
    {'Node_FK1': 'A4', 'Node_FK2': 'B4', 'Node_FK3': 'C4', 'Metrics': 'M3', 'Value': 20},
    {'Node_FK1': 'A4', 'Node_FK2': 'B4', 'Node_FK3': 'C4', 'Metrics': 'M4', 'Value': 49},
]

# --- Execution ---

# 1. Flatten individual hierarchies
flattened_h1 = flatten_hierarchy(hierarchy1_data)
flattened_h2 = flatten_hierarchy(hierarchy2_data)
flattened_h3 = flatten_hierarchy(hierarchy3_data)

# 2. Create lookup dictionaries for efficient merging
h1_lookup = create_lookup_dict(flattened_h1, 'Node_PK1')
h2_lookup = create_lookup_dict(flattened_h2, 'Node_PK2')
h3_lookup = create_lookup_dict(flattened_h3, 'Node_PK3')

# Combine lookups into a single dictionary for easier passing
all_hierarchy_lookups = {
    'Node_PK1': h1_lookup,
    'Node_PK2': h2_lookup,
    'Node_PK3': h3_lookup,
}

# 3. Merge and pivot the metrics hierarchy
final_flattened_output = merge_and_pivot_hierarchies(all_hierarchy_lookups, metrics_data)

print("\n--- Final Flattened Hierarchy ---")

# --- Dynamic Header and Printing Logic ---
# Determine min/max level and metric numbers dynamically from the first record
min_level_num = float('inf')
max_level_num = 0
min_metric_num = float('inf')
max_metric_num = 0

if final_flattened_output:
    # Iterate through all records to find the true min/max across the entire dataset
    # This handles cases where the first record might not contain all levels/metrics
    for record in final_flattened_output:
        for key in record.keys():
            level_match = re.match(r'Level(\d+)[A-C]', key) # Use re.match for start of string
            if level_match:
                level_num = int(level_match.group(1))
                min_level_num = min(min_level_num, level_num)
                max_level_num = max(max_level_num, level_num)

            metric_match = re.match(r'M(\d+)', key) # Use re.match for start of string
            if metric_match:
                metric_num = int(metric_match.group(1))
                min_metric_num = min(min_metric_num, metric_num)
                max_metric_num = max(max_metric_num, metric_num)

    # Handle cases where no levels or metrics were found
    if min_level_num == float('inf'): min_level_num = 1
    if max_level_num == 0: max_level_num = 0 # No levels found
    if min_metric_num == float('inf'): min_metric_num = 1
    if max_metric_num == 0: max_metric_num = 0 # No metrics found

# Define the header row dynamically
header_row = []
# Only add level headers if levels were detected
if max_level_num > 0:
    for i in range(min_level_num, max_level_num + 1):
        header_row.append(f'Level{i}A')
        header_row.append(f'Level{i}B')
        header_row.append(f'Level{i}C')
# Only add metric headers if metrics were detected
if max_metric_num > 0:
    for i in range(min_metric_num, max_metric_num + 1):
        header_row.append(f'M{i}')

print('\t'.join(header_row)) # Print the header row

# Print in a structured way to match your desired output order
for record in final_flattened_output:
    output_str = []
    # Only try to get level values if levels were detected
    if max_level_num > 0:
        for i in range(min_level_num, max_level_num + 1):
            output_str.append(f"{record.get(f'Level{i}A', '')}")
            output_str.append(f"{record.get(f'Level{i}B', '')}")
            output_str.append(f"{record.get(f'Level{i}C', '')}")
    # Only try to get metric values if metrics were detected
    if max_metric_num > 0:
        for i in range(min_metric_num, max_metric_num + 1):
            output_str.append(f"{record.get(f'M{i}', '')}")
    print('\t'.join(map(str, output_str))) # Use tab for separation, convert all to string

# To show the dictionary structure more clearly (for debugging/understanding)
# for record in final_flattened_output:
#     print(record)
